In [ ]:
### Visualize the Bayesian results and analyze each cluster.

In [ ]:
library(dplyr)
library(Matrix)
library(data.table)
library(Seurat)
library(ggplot2)
library(RColorBrewer)
library(cowplot)
library(future)
library(viridis)
library(SeuratObject)
library(tidyr)
plan("multicore", workers = 16)
options(future.globals.maxSize = 100 * 1024^3)
cols = c(brewer.pal(9, "Set1"),brewer.pal(8,"Set2")[1:8],brewer.pal(12,"Paired")[1:12],brewer.pal(8,"Dark2")[1:8],brewer.pal(8,"Accent"),brewer.pal(12, "Set3"),brewer.pal(9,"Pastel1"),brewer.pal(8,"Pastel2"))

In [ ]:
seuobj = readRDS("/data/Bin200.merge.debatch.rds")

In [ ]:
### add BayesSpace meta
n_clu = 16
meta = readRDS(paste0("/data/Bin200.debatch.bayes",n_clu,".outcome.rds"))

In [ ]:
meta = meta[,c("orig.ident","Bayes16")]
meta[1:2,]

In [ ]:
seuobj = AddMetaData(seuobj,meta)
seuobj$Bayes16 = as.character(seuobj$Bayes16)

In [ ]:
saveRDS(seuobj,"/data/Bin200.merge.debatch.rds")

In [ ]:
### save umap
options(repr.plot.width=15, repr.plot.height=5)  
pdf("/data/Bin200.bayes.cluster.UMAP.pdf",width=9)
for (i in c("16")){
    p = DimPlot(seuobj,reduction="stereo",group.by = paste0("Bayes",i),label = TRUE,label.size = 0.6,pt.size = 0.001,raster=FALSE,cols = cols)+coord_fixed()+ theme_void() +
          theme(axis.text = element_blank(), axis.ticks = element_blank(), axis.title = element_blank(), panel.background = element_rect(fill = "black"))
    print(p)
    }
dev.off()

In [ ]:
### split cluster
seuobj$Bayes16 = as.factor(seuobj$Bayes16)

In [ ]:
options(repr.plot.width=15, repr.plot.height=5)  
pdf("/data/Bin200.bayes.split.cluster.UMAP.pdf")
for (i in c("16")){
    p <-  ggplot(seuobj@meta.data,aes(as.numeric(coor_x), as.numeric(coor_y), fill= seuobj@meta.data[,paste0("Bayes",i)]))+geom_tile() + theme_classic()+ coord_fixed()+ scale_fill_manual(values =cols)+
            labs(fill=seuobj@meta.data[,paste0("Bayes",i)],x = "coor_x",y = "coor_y")+facet_wrap(. ~ seuobj@meta.data[,paste0("Bayes",i)],ncol = 4)+guides(fill=guide_legend(title=paste0("Bayes",i)))
    print(p)
    }
dev.off()

In [ ]:
### QC Vlnplot
DefaultAssay(seuobj) ="Spatial"
meta.all = seuobj@meta.data

In [ ]:
options(repr.plot.width=18, repr.plot.height=7)
pdf("/data/Bin200.bayes.cluster.QC.pdf",height=5,width=9)
for (i in c("16")){
    p = ggplot(meta.all,aes(x=meta.all[,paste0("Bayes",i)],y=nFeature_Spatial,fill=meta.all[,paste0("Bayes",i)]))+scale_fill_manual(values=cols)+
        theme_classic()+theme(panel.background = element_rect(fill="white"))+ 
        ylab("nFeature_Spatial")+xlab(paste0("Bayes",i))+RotatedAxis()+geom_boxplot(outlier.colour="white")
    print(p)
    }
dev.off()

In [ ]:
### del DEG
library(future)
options(future.globals.maxSize = 200 * 1024^3)
plan(multisession, workers=10)

DefaultAssay(seuobj)="Spatial"

In [ ]:
res.path = "/data/01.Bayes.clusterDEG/"

In [ ]:
for (res in c("16")){
    Idents(seuobj) = paste0("Bayes",res)
    scRNA1.markers <- FindAllMarkers(object = seuobj)
    write.csv(scRNA1.markers,paste0(res.path,"/","Bayes",res,".cluster.DEG.raw.csv"))
    ### filter
    scRNA1.markers = scRNA1.markers %>% subset(p_val_adj<0.05 & scRNA1.markers$avg_log2FC>0.25)
    write.csv(scRNA1.markers,paste0(res.path,"/","Bayes",res,".cluster.DEG.logFC0.25.P0.05.csv"))
    #### TOP10
    top10 <- scRNA1.markers %>% group_by(cluster) %>% top_n(n = 15, wt = avg_log2FC)
    write.csv(top10,paste0(res.path,"/","Bayes",res,".cluster.top15marker.csv"))
    }

In [ ]:
### marker dotplot
features = c("RELN","FABP7",  ##l1
             "C1QL2","LAMP5","VIP", ##l2
             "COL5A2", ##l3
             "TSHZ2","RORB", ##l4
             "HTR2A","HS3ST2","PCP4","HTR2C",  ##l5
             "NTNG2","THEMIS","SYT6","SEMA3E","CCN2","KRT17",  ##l6
             "MBP",##WM
             "MOBP","PLP1","OLIG1","OLIG2", ##Oligo
             "HBB","HBA2","HBA1",  ##blood
            "MT-TM","MT-TG","MT-ATP8") ##WM) 

In [ ]:
options(repr.plot.width=10, repr.plot.height=5)   
p1 =  DotPlot(seuobj,group.by="Bayes20",features = features)+theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5))+
        scale_color_gradientn(colours = viridis::viridis(20), guide = guide_colorbar(ticks.colour = "black",frame.colour = "black"),name = "Average\nexpression")
print(p1)
ggsave(plot=p1,"/data/Bin200.marker.dotplot.pdf",width=9,height=6)